# Chapitre 18 - Problèmes en grande dimension ($p \gg N$)
## Cours 1 : pourquoi $p \gg N$ est difficile, LDA diagonale et centroïdes réduits

Ce chapitre traite des problèmes de prédiction où le nombre de variables $p$ est
**beaucoup plus grand** que le nombre d'observations $N$. C'est la situation typique
de la génomique : quelques dizaines de patients, des milliers de gènes mesurés.

Dans ce premier notebook :

1. une simulation (figure 18.1 du livre) qui montre que, lorsque $p \gg N$,
   **« moins ajuster, c'est mieux »** : il faut régulariser fortement ;
2. l'**analyse discriminante linéaire diagonale** (règle « d'indépendance ») ;
3. les **centroïdes réduits** (*nearest shrunken centroids*, NSC), appliqués aux
   données de micro-puces NCI (analogue de la figure 18.4).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl import ch18_grande_dimension as g

## 1. Notations et rappels

On dispose de $N$ observations $(x_i, y_i)$, avec $x_i \in \mathbb{R}^p$, rangées dans
la matrice $\mathbf{X}$ de taille $N \times p$ (une ligne par observation).

Lorsque $p > N$, la matrice $\mathbf{X}^T\mathbf{X}$ (de taille $p\times p$) est de rang
au plus $N < p$ : elle est **singulière**. Les moindres carrés ordinaires
$\hat\beta = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T y$ ne sont donc pas définis, et
il existe une infinité de vecteurs $\beta$ qui interpolent exactement les données
(voir l'exercice 18.7).

**Rappel : la régression ridge** (section 3.4.1 du livre). Pour $\lambda > 0$, on
minimise $\|y - \mathbf{X}\beta\|^2 + \lambda\|\beta\|^2$, dont la solution est
$$\hat\beta_\lambda = (\mathbf{X}^T\mathbf{X} + \lambda I_p)^{-1}\mathbf{X}^T y .$$
La matrice $\mathbf{X}^T\mathbf{X} + \lambda I_p$ est inversible car elle est symétrique
et ses valeurs propres sont $\geq \lambda > 0$.

Si $\mathbf{X} = \mathbf{U}\mathbf{D}\mathbf{V}^T$ est la SVD (valeurs singulières
$d_1 \ge \dots \ge d_r$), les **degrés de liberté effectifs** (éq. (3.50) du livre) valent
$$\mathrm{df}(\lambda) = \operatorname{tr}\!\big[\mathbf{X}(\mathbf{X}^T\mathbf{X}+\lambda I)^{-1}\mathbf{X}^T\big]
  = \sum_{j=1}^{r} \frac{d_j^2}{d_j^2+\lambda}.$$
Ils décroissent de $r = \operatorname{rang}(\mathbf{X})$ (pour $\lambda \to 0$) à $0$
(pour $\lambda\to\infty$) : c'est une mesure plus parlante que $\lambda$ de la
complexité de l'ajustement.

## 2. Une simulation : « moins ajuster, c'est mieux » (figure 18.1)

**Protocole** (reformulé d'après la section 18.1) :

- $N = 100$ observations, $p \in \{20, 100, 1000\}$ variables gaussiennes centrées
  réduites, de **corrélation deux à deux $0.2$** ;
- $Y = \sum_{j} X_j\beta_j + \sigma\varepsilon$ (éq. (18.1)), avec
  $\varepsilon \sim \mathcal N(0,1)$ et des coefficients $\beta_j \sim \mathcal N(0,1)$
  tirés à nouveau pour chaque jeu de données ;
- $\sigma$ est choisi pour que le **rapport signal sur bruit**
  $\operatorname{Var}[\mathbb E(Y\mid X)]/\sigma^2$ vaille $2$ ;
- on ajuste une ridge avec $\lambda \in \{0.001,\ 100,\ 1000\}$.

**Construire des variables de corrélation 0.2.** Soient $Z_1,\dots,Z_p$ et $W$
indépendantes $\mathcal N(0,1)$, et $X_j = \sqrt{0.8}\,Z_j + \sqrt{0.2}\,W$. Alors,
par indépendance,
$$\operatorname{Var}(X_j) = 0.8 + 0.2 = 1, \qquad
  \operatorname{Cov}(X_j, X_k) = \operatorname{Cov}(\sqrt{0.2}W, \sqrt{0.2}W) = 0.2 \quad (j\neq k).$$
La matrice de covariance est donc $\Sigma = 0.8\, I_p + 0.2\, \mathbf 1\mathbf 1^T$.

**Erreur de test exacte.** Pour un nouveau point $(x_0, Y_0)$ indépendant de
l'échantillon d'apprentissage, avec $Y_0 = x_0^T\beta + \sigma\varepsilon_0$ et
$x_0 \sim \mathcal N(0, \Sigma)$, on a, conditionnellement à $\hat\beta$ :
$$\begin{aligned}
\mathbb E\big[(Y_0 - x_0^T\hat\beta)^2\big]
  &= \mathbb E\big[(\sigma\varepsilon_0 - x_0^T(\hat\beta - \beta))^2\big] \\
  &= \sigma^2 + \mathbb E\big[(x_0^T e)^2\big] \qquad (e = \hat\beta-\beta,\ \text{le terme croisé est nul car } \varepsilon_0 \perp x_0 \text{ et } \mathbb E\varepsilon_0 = 0)\\
  &= \sigma^2 + e^T\,\mathbb E[x_0x_0^T]\,e = \sigma^2 + e^T\Sigma e .
\end{aligned}$$
L'**erreur relative** (erreur de test divisée par l'erreur de Bayes $\sigma^2$) vaut
donc $1 + e^T\Sigma e/\sigma^2$ ; pas besoin d'échantillon test. De même,
$\operatorname{Var}[\mathbb E(Y\mid X)] = \beta^T\Sigma\beta$, d'où
$\sigma^2 = \beta^T\Sigma\beta/2$.

> **Remarque.** Les variables étant centrées, on ajuste la ridge **sans intercept**.
> Pour $p = 1000$, on calcule $\hat\beta_\lambda$ par la SVD de $\mathbf X$
> (formule (18.15), vue au cours 2), ce qui coûte $O(N^2p)$ au lieu de $O(p^3)$.

In [ ]:
def simulate_ridge(p, lams, n_rep, rng, N=100):
    """Erreurs relatives, ddl et nombre de coefficients univariés significatifs."""
    rel, df, nsig = [], [], []
    for _ in range(n_rep):
        X = np.sqrt(0.8) * rng.randn(N, p) + np.sqrt(0.2) * rng.randn(N, 1)
        beta = rng.randn(p)
        quad = lambda e: 0.8 * (e ** 2).sum(-1) + 0.2 * e.sum(-1) ** 2   # e^T Sigma e
        sigma2 = quad(beta) / 2
        y = X @ beta + np.sqrt(sigma2) * rng.randn(N)
        B = g.ridge_svd(X, y, lams)            # (len(lams), p)
        rel.append(1 + quad(B - beta) / sigma2)
        df.append(g.ridge_df(X, lams))
        nsig.append(np.sum(np.abs(g.univariate_scores(X, y)) >= 2))
    return np.array(rel), np.array(df), np.array(nsig)

rng = np.random.RandomState(0)
lams = np.array([0.001, 100.0, 1000.0])
ps = [20, 100, 1000]
res = {p: simulate_ridge(p, lams, 100, rng) for p in ps}
for p in ps:
    rel, df, nsig = res[p]
    print(f"p = {p:4d} : ddl moyens = {np.round(df.mean(0), 1)}, "
          f"erreur relative médiane = {np.round(np.median(rel, 0), 2)}, "
          f"coef. univariés significatifs ≈ {nsig.mean():.0f}")